# LifeMate — GitHub → Colab → public submission
Choose **Runtime → Change runtime type → GPU**, then run the cells in order.
This notebook uses Colab's Python runtime and its installed CUDA PyTorch. No virtual environment or Drive mount is used.
The dataset is downloaded with gdown. Results are saved under `/content/lifemate_runs`; download them before the runtime is deleted.


In [ ]:
# 1. Clone the repository and use its project directory.
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/votaquangnhat/LifeMate-Baseline.git"
PROJECT = Path("/content/lifemate")
if not PROJECT.exists():
    !git clone {REPO_URL} {PROJECT}
    assert _exit_code == 0, "Git clone failed. Check repository access."
else:
    print("Using existing clone:", PROJECT)
os.chdir(PROJECT)
assert (PROJECT / "requirements-colab.txt").is_file(), "Push the updated code, including requirements-colab.txt, to GitHub first."
!nvidia-smi
assert _exit_code == 0, "Select a GPU runtime first."


In [ ]:
# 2. Download dataset
!pip -q install gdown
!gdown 1f5Vdnm3fhLdBYZFmRPPYvKscmFmbozmQ -O /content/train+public_test.zip

from pathlib import Path
ARCHIVE = Path("/content/train+public_test.zip")
assert _exit_code == 0 and ARCHIVE.is_file(), "Dataset download failed; check the Drive file's access/quota."


In [ ]:
# 3. Extract the dataset on Colab's local disk.
import shutil
DATA_DIR = Path("/content/lifemate-data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not list(DATA_DIR.glob("**/public_test/public_test.jsonl")):
    shutil.unpack_archive(str(ARCHIVE), str(DATA_DIR))
manifest_paths = list(DATA_DIR.glob("**/public_test/public_test.jsonl"))
assert len(manifest_paths) == 1, f"Expected one public_test manifest; found {manifest_paths}"
DATA_ROOT = manifest_paths[0].parent.parent
assert (DATA_ROOT / "train/train.jsonl").is_file(), "The ZIP must contain train and public_test."
print("Data root:", DATA_ROOT)


In [ ]:
# 4. Install packages into the CURRENT Colab runtime. Keep Colab's CUDA PyTorch.
!{sys.executable} -m pip install -q -r requirements-colab.txt
assert _exit_code == 0, "Dependency installation failed. Inspect pip's error above."

import torch
from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "This pipeline requires PyTorch >=2.6; use an up-to-date Colab runtime."
assert torch.cuda.is_available(), "Select a GPU runtime."
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
# If Colab explicitly requests a session restart, restart, then rerun cells 1 onward.


In [ ]:
# 5. Download the baseline model weights automatically.
!{sys.executable} scripts/download_models.py --profile baseline
assert _exit_code == 0, "Model download failed. Fix it and rerun this cell."


In [ ]:
# 6. Configure dataset paths and prepare training-only retrieval examples.
import json
cfg = json.loads((PROJECT / "configs/colab.json").read_text(encoding="utf-8-sig"))
cfg["data_root"] = str(DATA_ROOT)
CONFIG = PROJECT / "configs/colab-runtime.json"
CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
!{sys.executable} -m baseline.prepare --data-root "{DATA_ROOT}"
assert _exit_code == 0, "Data preparation failed."
!{sys.executable} -m baseline.doctor --config "{CONFIG}" --split public_test
assert _exit_code == 0, "Preflight failed."


In [ ]:
# 7. Test ONE training audio before running the full public split.
from datetime import datetime, timezone
RUN_DIR = Path("/content/lifemate_runs") / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
RUN_DIR.mkdir(parents=True, exist_ok=False)
SMOKE = RUN_DIR / "colab_smoke.jsonl"
!{sys.executable} -m baseline.run --config "{CONFIG}" --split train --ids artifacts/splits/dev_ids.json --limit 1 --seconds 120 --output "{SMOKE}" --verbose
assert _exit_code == 0, "Smoke inference failed. Review the error/diagnostics."
!{sys.executable} -m baseline.inspect_run "{SMOKE}" --require-complete
assert _exit_code == 0, "The smoke run did not finish normally."
print("Output directory:", RUN_DIR)


## Full public run
Continue after the smoke run completes. The next cell runs every public sample with the 28-second cooperative budget.
Results are saved locally under `/content/lifemate_runs`. Download the completed result in cell 10 before the runtime is deleted.
A runtime disconnect may leave a partial file; full-split validation must pass before submission. There is no automatic resume.


In [ ]:
# 8. Generate ALL public_test predictions. No sample limit here.
SUBMISSION = RUN_DIR / "submission.jsonl"
!{sys.executable} -m baseline.run --config "{CONFIG}" --split public_test --output "{SUBMISSION}"
assert _exit_code == 0, "Inference reported failures. Inspect diagnostics before submitting."


In [ ]:
# 9. Validate the entire submission and review timeouts/errors.
!{sys.executable} -m baseline.validate_submission "{SUBMISSION}" --data-root "{DATA_ROOT}" --split public_test
assert _exit_code == 0, "Submission validation failed. Do not submit a partial/invalid file."
!{sys.executable} -m baseline.inspect_run "{SUBMISSION}"
assert _exit_code == 0, "Could not inspect the run."
print("Submit THIS file:", SUBMISSION)


In [ ]:
# 10. Download submission.jsonl, then upload it to the competition portal.
from google.colab import files
files.download(str(SUBMISSION))


## Later: teammate checkpoints
Download/upload the checkpoints into Colab's local disk and change `cfg["asr"]["path"]` and `cfg["llm"]["adapter"]` in cell 6.
See `TEAM_HANDOFF.md` for checkpoint formats. Use a new output directory for each experiment.

If a cell fails, fix it before continuing. To start another experiment, rerun cell 7 to create a new result directory.
If Colab requests a restart after installation, restart the session and rerun cells 1 onward. The runtime's Python executes all commands; no separate virtual environment is created.
The Colab CUDA PyTorch version is retained; GPU performance must be measured on the actual runtime.
